<a href="https://colab.research.google.com/github/celinehsu/Complete-Python-3-Bootcamp/blob/master/FindSimilarJobs_(Careers_Gov).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install required helper packages (takes ~20 seconds)
!pip install pandas requests sentence-transformers scikit-learn

import requests
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# PASTE YOUR JOB DESCRIPTION INSIDE THE TRIPLE QUOTES BELOW:
# ---------------------------------------------------------------------------
MY_JOB_DESCRIPTION = """
The Senior Manager in Centre for Information Management (CIM) will lead a team to support DSTA’s record management policies, processes and systems in accordance to Government Instruction Manual on Records Management and DSTA records management framework, as well as drive process reviews and digitalization to ensure that DSTA’s record management system continue meet DSTA’s evolving needs.



Job scope:

Lead and manage DSTA Enterprise Content Management and records repositories to comply with DSTA Governance Framework and align with international and national standards and practices for optimal records management.


Plan and develop DSTA Records Management Roadmap for digital/digitised records including storage, accessibility and disposal.


Formalise holistic governance framework, policies and processes for continual improvement in management of DSTA enterprise content (i.e. digital/digitised records). Evaluate, test and implement the system capabilities.


Review, recommend and drive digitalisation initiatives with key stakeholders (e.g. DSTA management/entities, MINDEF/SAF, external agencies) in records management to meet organisation's evolving business needs.


Support DSTA Chief Record Officer in Records Management to ensure compliance with Government’s Instruction Manual requirements.
"""
# ---------------------------------------------------------------------------

print("Fetching active Careers@Gov postings...")
url = "https://raw.githubusercontent.com/opengovsg/careersgovsg-jobs-data/main/data/job-listings.json"
response = requests.get(url)
jobs = response.json()
df = pd.DataFrame(jobs)

def get_url(row):
    plat, jid, pno = row.get("platform", ""), row.get("jobId", ""), row.get("postingNo", "")
    if plat == "hrp":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}/{pno}"
    elif plat == "greenhouse":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}?gh_jid={jid}"
    elif plat == "workable":
        return f"https://jobs.careers.gov.sg/jobs/{plat}/{jid}/{pno}"
    return "https://jobs.careers.gov.sg"

def clean_html(text):
    if not isinstance(text, str):
        return ""
    return " ".join(re.sub(r"<.*?>", " ", text).split())

df["link"] = df.apply(get_url, axis=1)

# Combine fields for semantic comparison
df["full_text"] = (
    "Title: " + df["jobTitle"].fillna("") + "\n" +
    "Agency: " + df["agency"].fillna("") + "\n" +
    "Responsibilities: " + df.get("jobResponsibilities", pd.Series([""]*len(df))).fillna("").apply(clean_html) + "\n" +
    "Requirements: " + df.get("jobRequirements", pd.Series([""]*len(df))).fillna("").apply(clean_html)
)

print(f"Loaded {len(df)} postings. Loading AI matching model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

target_emb = model.encode([MY_JOB_DESCRIPTION])
job_embs = model.encode(df["full_text"].tolist(), show_progress_bar=True)

df["match_score_%"] = (cosine_similarity(target_emb, job_embs)[0] * 100).round(1)

# Pick top 15 matches
top_matches = df.sort_values(by="match_score_%", ascending=False).head(15)

print("\n" + "="*80)
print("TOP MATCHING JOBS ON CAREERS@GOV:")
print("="*80)

for idx, (_, row) in enumerate(top_matches.iterrows(), 1):
    print(f"\n{idx}. [{row['match_score_%']}% Match] {row['jobTitle']}")
    print(f"   Agency: {row.get('agency', 'N/A')}")
    print(f"   Link:   {row['link']}")

# Save as an Excel-friendly file
top_matches[["jobTitle", "agency", "match_score_%", "link"]].to_csv("my_matched_jobs.csv", index=False)
print("\n Done! A file named 'my_matched_jobs.csv' has been generated.")

Fetching active Careers@Gov postings...
Loaded 2105 postings. Loading AI matching model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]


TOP MATCHING JOBS ON CAREERS@GOV:

1. [96.0999984741211% Match] Senior Manager, Centre for Info Mgmt (CIM), Corp Planning & Services
   Agency: Defence Science and Technology Agency
   Link:   https://jobs.careers.gov.sg/jobs/hrp/18124292/005056a3-d347-1fe1-b0c4-843f00cfc2bf

2. [73.5999984741211% Match] (Senior) Business Partner, Administration
   Agency: Enterprise Singapore
   Link:   https://jobs.careers.gov.sg/jobs/hrp/18198326/005056a3-d347-1fe1-b5f8-853b256b42c5

3. [69.9000015258789% Match] Manager/Senior Manager, Data Management (DSAD) [Contract till 30 Dec 2027]
   Agency: Skills and Workforce Development Agency
   Link:   https://jobs.careers.gov.sg/jobs/hrp/18105553/005056a3-d347-1fe1-b6aa-cbcfe5a822c5

4. [68.5% Match] Assistant Archivist/Archivist, Records Management (Digital Preservation)
   Agency: National Library Board
   Link:   https://jobs.careers.gov.sg/jobs/hrp/17918944/005056a3-53e2-1fd1-b4e1-a164b3c9b40d

5. [67.80000305175781% Match] [DGDD] Deputy Director (E